<a href="https://colab.research.google.com/github/Vga18/diplomado/blob/main/Proyecto_diplomado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cont-AI**

## **Planteamiento del problema**

El desarrollo de pólizas es un proceso automático y tardado, ya que requiere tener un criterio para decidir que gastos van su cuenta contable

Plataforma de usuario para agilizar la capturación de pólizas de egresos

Acotado a las facturas XML-CFDI recibidas, y al catalogo de cuentas contables que maneja el Sistema Aspel COI versión 9



## Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


pd.set_option("display.max_columns",200)
pd.set_option("display.max_rows",100)

## Funciones Auxiliares

## Datos

In [2]:
catalogo = pd.read_csv(r"/content/Datos/catalogo de cuentas contables.csv")
polizas22 = pd.read_csv(r"/content/Datos/polizas22.csv")
polizas23 = pd.read_csv(r"/content/Datos/polizas23.csv")
polizas24 = pd.read_csv(r"/content/Datos/polizas24.csv")
polizas25 = pd.read_csv(r"/content/Datos/polizas25.csv")
polizas26 = pd.read_csv(r"/content/Datos/polizas26.csv")

## Ingenieria

In [3]:
catalogo.tail(3)

,NUM_CTA,STATUS,TIPO,NOMBRE,DEPTSINO,BANDMULTI,BANDAJT,CTA_PAPA,CTA_RAIZ,NIVEL,CTA_COMP,NATURALEZA,RFC,CODAGRUP,CAPTURACHEQUE,CAPTURAUUID,BANCO,CTABANCARIA,CAPCHEQTIPOMOV,NOINCLUIRXML,IDFISCAL,ESFLUJODEEFECTIVO,BANCOEXTRANJERO,RFCFLUJO
305,117000300300000000003,A,D,VICENTE VEGA RAMIREZ,N,1,N,117000300000000000002,117000000000000000001,3,NaN,0,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN
306,212000100200000000003,A,D,FRANCISCO D. VEGA HERNANDEZ,N,1,N,212000100000000000002,212000000000000000001,3,NaN,1,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN
307,111000200000000000002,A,D,TAG CUENTA.0002706454,N,1,N,111000000000000000001,111000000000000000001,2,0.0,0,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN


Ya que el objetivo es la clasificación de numero de cuentas se eligira ciertas variables

In [4]:
catalogo = catalogo[["NUM_CTA","STATUS","TIPO","NOMBRE","NIVEL","NATURALEZA"]]
catalogo = catalogo[catalogo['NIVEL'] != 1]
catalogo.sample(5)

,NUM_CTA,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA
93,218000200000000000002,A,D,IEPS COBRADO,2,1
100,220000100000000000002,A,D,CREDITO #,2,1
43,131000200000000000002,A,D,EDIFICIOS,2,0
83,216000200000000000002,A,D,PROVISION DE VACACIONES POR PAGAR,2,1
181,620000100000000000002,A,D,SUELDOS Y SALARIOS,2,0


In [5]:
display(polizas26.head(5))

,TIPO_POLI,NUM_POLIZ,NUM_PART,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,MONTOMOV,NUMDEPTO,TIPCAMBIO,CONTRAPAR,ORDEN,CCOSTOS,CGRUPOS,IDINFADIPAR,IDUUID
0,Eg,1,1.0,1,2026,610004200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",D,994.83,0,1.0,0,1,0,0,-1,-1
1,Eg,1,2.0,1,2026,120000100000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",D,159.17,0,1.0,0,2,0,0,-1,-1
2,Eg,1,3.0,1,2026,111000200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",H,1154.00,0,1.0,0,3,0,0,-1,-1
3,Eg,2,1.0,1,2026,610004200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-1760426, CONCESIONARIA MEXIQ...",D,200.00,0,1.0,0,1,0,0,-1,-1
4,Eg,2,2.0,1,2026,120000100000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-1760426, CONCESIONARIA MEXIQ...",D,32.00,0,1.0,0,2,0,0,-1,-1


In [6]:
variables_destables = ["TIPO_POLI","PERIODO","EJERCICIO","NUM_CTA","FECHA_POL","CONCEP_PO","DEBE_HABER"]

Polizas22 = polizas22[variables_destables]
Polizas23 = polizas23[variables_destables]
Polizas24 = polizas24[variables_destables]
Polizas25 = polizas25[variables_destables]
polizas26 = polizas26[variables_destables]

polizas = pd.concat([Polizas22,Polizas23,Polizas24,Polizas25,polizas26])

In [7]:
polizas = polizas[polizas["DEBE_HABER"] == "D"]
polizas = polizas[polizas["CONCEP_PO"] != "CANCELADO"]

In [8]:
polizas.isnull().sum().sort_values(ascending=False)

,0
CONCEP_PO,4
PERIODO,0
TIPO_POLI,0
EJERCICIO,0
NUM_CTA,0
FECHA_POL,0
DEBE_HABER,0


In [9]:
polizas = polizas.dropna()

In [10]:
polizas = polizas[polizas['CONCEP_PO'].str.contains("FACTURA")]

In [11]:
polizas

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER
5,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D
6,Dr,2,2022,120000100000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D
10,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D
11,Eg,2,2022,120000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D
14,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D
...,...,...,...,...,...,...,...
144,Eg,1,2026,112000100100000000003,2026-01-28 00:00:00.000,"PAGO FACTURA NO., AUTOZONE",D
145,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D
146,Eg,1,2026,120000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D
149,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.GCM-049355, GRUPO DAISTYTEK",D


In [12]:
pol_cat = pd.merge(polizas,catalogo,how="left",left_on="NUM_CTA",right_on="NUM_CTA")

In [13]:
pol_cat

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA
0,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,TELEFONOS,2,0
1,Dr,2,2022,120000100000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,IVA ACREDITABLE PAGADO,2,0
2,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1
3,Eg,2,2022,120000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,IVA ACREDITABLE PAGADO,2,0
4,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4741,Eg,1,2026,112000100100000000003,2026-01-28 00:00:00.000,"PAGO FACTURA NO., AUTOZONE",D,A,D,CTA: 0118116830,3,0
4742,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D,A,D,PROVEEDOR #,2,1
4743,Eg,1,2026,120000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D,A,D,IVA ACREDITABLE PAGADO,2,0
4744,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.GCM-049355, GRUPO DAISTYTEK",D,A,D,PROVEEDOR #,2,1
